# Reinforcement Learning with CartPole
## Training an AI Agent to Balance a Pole using PPO

---

This notebook walks through a complete Reinforcement Learning project using the **CartPole-v1** environment from Gymnasium and the **PPO** (Proximal Policy Optimization) algorithm from Stable-Baselines3.

The agent starts with no knowledge of the task and learns purely through trial and error, receiving a reward of +1 for every timestep the pole stays upright.

**Libraries used:** `gymnasium`, `stable-baselines3`, `numpy`, `matplotlib`

---
## Section 01 -- Installation

Install the required libraries before running any other cell. This only needs to be done once per environment.

In [ ]:
pip install gymnasium
pip install stable-baselines3
pip install matplotlib numpy

---
## Section 02 -- Exploring the CartPole Environment

Before training anything, we first need to understand the environment: what the agent sees (observations), what it can do (actions), and how episodes start and end.

In [ ]:
import gymnasium as gym
import numpy as np

# Create the CartPole environment
# render_mode="human" opens a graphical window (optional during testing)
env = gym.make("CartPole-v1")

# Reset the environment -- required before starting any episode
# obs  = the initial observation (starting state)
# info = additional information (usually empty at the start)
obs, info = env.reset()

print("Initial observation:", obs)
print("Type:", type(obs))
print("Shape:", obs.shape)

The observation is a NumPy array of 4 values describing the full state of the system:

| Index   | Variable             | Range          | Meaning                                                                 |
|---------|----------------------|----------------|-------------------------------------------------------------------------|
| obs[0]  | Cart position        | -2.4 to +2.4   | Where the cart is on the track. Episode ends if out of bounds.          |
| obs[1]  | Cart velocity        | -inf to +inf   | How fast the cart moves. Positive = right, negative = left.             |
| obs[2]  | Pole angle           | -12deg to +12deg | Tilt of the pole from vertical. Episode ends if beyond +/-12 degrees. |
| obs[3]  | Pole angular velocity| -inf to +inf   | How fast the pole is tilting. Key for anticipating a fall.              |

In [ ]:
# The observation space describes the possible values the agent can see
print("Observation space:", env.observation_space)
# Result: Box(-inf, inf, (4,), float32)
# Box = continuous values, (4,) = 4 variables, float32 = floating point numbers

# The action space describes the available actions
print("Action space:", env.action_space)
# Result: Discrete(2)
# Discrete(2) = two possible actions: 0 (push left) or 1 (push right)

print("Number of actions:", env.action_space.n)

---
## Section 03 -- Random Agent (Baseline)

Before training anything, we test a completely random agent. This gives us a **baseline**: a reference score to measure how much the trained agent improves.

In [ ]:
import numpy as np

# We measure scores over 10 episodes to get an average
scores_random = []

for episode in range(10):

    obs, info = env.reset()  # Reset the environment
    score = 0                # Reward counter for this episode
    done = False

    while not done:

        # Choose an action completely at random (0 or 1)
        action = env.action_space.sample()

        # Execute the action in the environment
        # obs        = new state after the action
        # reward     = reward received (+1 if pole is upright, 0 otherwise)
        # terminated = True if the pole fell or the cart went out of bounds
        # truncated  = True if the 500-step limit was reached
        obs, reward, terminated, truncated, info = env.step(action)

        score += reward         # Accumulate the reward
        done = terminated or truncated

    scores_random.append(score)
    print(f"Episode {episode+1} -- Score: {score}")

print(f"\nAverage score (random agent): {np.mean(scores_random):.1f}")

---
## Section 04 -- PPO Algorithm: Proximal Policy Optimization

PPO is the algorithm we will use to train the agent. It was developed by OpenAI in 2017 and is one of the most widely used RL algorithms in both research and industry.

The **policy** is the agent's strategy: a neural network that takes the 4 observation values as input and returns a probability distribution over the two possible actions. PPO improves this policy progressively after each batch of collected experience, while enforcing a stability constraint: the new policy cannot deviate too far from the old one at each update step.

---
## Section 05 -- Training the Agent

We create the PPO model and launch training. Stable-Baselines3 automatically configures the neural network architecture, learning rate, and all default hyperparameters -- these defaults are well-tuned for CartPole.

Training 50,000 timesteps takes roughly 1-2 minutes on a standard CPU.

In [ ]:
from stable_baselines3 import PPO

# Re-create a clean environment for training
env = gym.make("CartPole-v1")

# Create the PPO model
# "MlpPolicy" : fully connected neural network (Multi-Layer Perceptron)
# env         : the environment in which the agent will learn
# verbose=1   : print training logs to the terminal
model = PPO(
    "MlpPolicy",
    env,
    verbose=1
)

# Start training
# total_timesteps : total number of experience timesteps to collect
# 50,000 steps is sufficient for CartPole -- a few minutes on CPU
model.learn(total_timesteps=50_000)

print("Training complete.")

In [ ]:
# Save the model to disk
# This creates a ppo_cartpole.zip file containing the network weights
model.save("ppo_cartpole")
print("Model saved: ppo_cartpole.zip")

# To reload it later:
# model = PPO.load("ppo_cartpole")

---
## Section 06 -- Evaluation: Trained Agent vs. Random Agent

We now evaluate the trained agent using `model.predict()` instead of random sampling. The `deterministic=True` parameter means we always take the most likely action -- no random exploration. During evaluation, we want to see the agent's best performance.

In [ ]:
import numpy as np

scores_trained = []

for episode in range(10):

    obs, info = env.reset()
    score = 0
    done = False

    while not done:

        # model.predict() returns the action chosen by the trained policy
        # deterministic=True : always take the most likely action (no randomness)
        action, _ = model.predict(obs, deterministic=True)

        obs, reward, terminated, truncated, info = env.step(action)
        score += reward
        done = terminated or truncated

    scores_trained.append(score)
    print(f"Episode {episode+1} -- Score: {score}")

print(f"\nAverage score (random agent)  : {np.mean(scores_random):.1f}")
print(f"Average score (trained agent) : {np.mean(scores_trained):.1f}")

---
## Section 07 -- Visualizing Learning Progress

We evaluate the trained agent over 100 episodes and plot the score distribution. The chart shows:
- **Raw scores** per episode (light, transparent points showing variability)
- **Rolling average** over 10 episodes (smoothed trend line)
- **Random agent baseline** (dashed reference line)

The gap between the dashed line and the solid curve represents everything the RL training achieved.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Evaluate the trained agent over 100 episodes to see the score distribution
scores = []

for episode in range(100):

    obs, info = env.reset()
    score = 0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        score += reward
        done = terminated or truncated

    scores.append(score)

# Compute rolling average over 10 episodes to smooth the curve
rolling_average = [
    np.mean(scores[max(0, i-9):i+1])
    for i in range(len(scores))
]

In [ ]:
plt.figure(figsize=(10, 5))

# Raw score for each episode (light points)
plt.plot(scores, alpha=0.3, color="#1a1916", label="Score per episode")

# Rolling average (main curve)
plt.plot(rolling_average, color="#1a1916", linewidth=2, label="Rolling average (10 episodes)")

# Reference line: random agent average score
plt.axhline(
    y=np.mean(scores_random),
    color="#c8b98a",
    linestyle="--",
    linewidth=1.5,
    label=f"Random agent (avg. {np.mean(scores_random):.0f})"
)

plt.xlabel("Episode")
plt.ylabel("Score (cumulative reward)")
plt.title("PPO Trained Agent Performance -- CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Average score over 100 episodes: {np.mean(scores):.1f}")
print(f"Minimum score: {min(scores):.0f}")
print(f"Maximum score: {max(scores):.0f}")

---
## Section 08 -- Conclusion and Next Steps

The project is complete. Here is a summary of what was built:

- Understood the core RL concepts: agent, environment, actions, rewards, policy, and the learning loop.
- Explored the CartPole-v1 environment: 4 observation variables, 2 possible actions, +1 reward per timestep the pole stays upright.
- Tested a random agent as a baseline: scores between 10 and 30.
- Trained a PPO agent over 50,000 timesteps: consistently reaches the maximum score of 500.
- Visualized the performance gap between the random agent and the trained agent.

**Going further -- just change one line:**

| Environment         | Task                                    |
|---------------------|-----------------------------------------|
| `LunarLander-v2`   | Train a rocket to land                  |
| `MountainCar-v0`   | Train a car to climb a hill             |
| Atari environments | Train on real video games               |

The training pipeline (environment creation, PPO model, `model.learn()`, evaluation loop) remains exactly the same across all of these environments.